# 03 — Facilities Registry Silver

Builds `public_health.silver.facilities_registry_silver` — one row per facility establishment, keyed by `facility_key`.

### Grain & Key
- 1 row per facility establishment (not per licence or service)
- Key: `facility_key` (from `facilities_id_map_silver`)
- Write: MERGE (observation always, business fields on change)

### Sources
- NHFR: `public_health.bronze.facilities_nhfr_bronze` (1,359 hospitals) — facility type, ownership, beds, licensing, PSGC codes
- HFSRB hospitals: `public_health.bronze.facilities_hfsrb_hospitals_bronze` (1,336 hospitals) — ABC, hospital class, validity dates
- ID map: `public_health.silver.facilities_id_map_silver` (2,695 rows, 1,534 unique facility_keys)
- Geography: `public_health.silver.geography_silver` (43,762 PSGC codes)

### Key Rules (plan v1.4, sections 2-5)
- Hospital-based services (dialysis, blood) are attributes of the hospital row
- Bed measures: `hfsrb_abc` (authorized), `nhfr_reported_beds`, `analysis_beds = coalesce(hfsrb_abc, nhfr_reported_beds)`
- Geography: PSGC from NHFR `City/Municipality_PSGC`, HFSRB inherits via matched `facility_key`
- Presence: `is_currently_listed_nhfr/hfsrb = true` for latest snapshot; v1: no closure inference
- Lineage: combined-row pattern with `nhfr` and `hfsrb` sides (section 5.3)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import uuid

ID_MAP_TABLE = "public_health.silver.facilities_id_map_silver"
NHFR_BRONZE = "public_health.bronze.facilities_nhfr_bronze"
HFSRB_BRONZE = "public_health.bronze.facilities_hfsrb_hospitals_bronze"
GEOGRAPHY_SILVER = "public_health.silver.geography_silver"
TARGET_TABLE = "public_health.silver.facilities_registry_silver"
SILVER_BUILD_ID = str(uuid.uuid4())

print(f"Config loaded. Silver build ID: {SILVER_BUILD_ID}")
print(f"ID map       : {ID_MAP_TABLE}")
print(f"NHFR bronze  : {NHFR_BRONZE}")
print(f"HFSRB bronze : {HFSRB_BRONZE}")
print(f"Geography    : {GEOGRAPHY_SILVER}")
print(f"Target table : {TARGET_TABLE}")

In [0]:
# Read the id map and split into NHFR and HFSRB sides
id_map = spark.table(ID_MAP_TABLE)

# NHFR side: one row per NHFR facility_key
nhfr_side = (
    id_map
    .filter(F.col("_source_key") == "doh_nhfr")
    .select(
        F.col("facility_key"),
        F.col("nhfr_id"),
        F.col("facility_name").alias("nhfr_facility_name"),
        F.col("municipality_name").alias("nhfr_municipality_name"),
        F.col("source_id_raw").alias("nhfr_source_id_raw"),
    )
)

# HFSRB side: one row per HFSRB facility_key (exclude duplicate_of rows)
hfsrb_side = (
    id_map
    .filter(F.col("_source_key") == "doh_hfsrb_hospitals")
    .filter(F.col("disposition") != "duplicate_of")
    .select(
        F.col("facility_key"),
        F.col("source_record_id").alias("hfsrb_source_record_id"),
        F.col("facility_name").alias("hfsrb_facility_name"),
        F.col("municipality_name").alias("hfsrb_municipality_name"),
        F.col("match_method"),
        F.col("match_status"),
        F.col("nhfr_id").alias("hfsrb_matched_nhfr_id"),
    )
)

nhfr_count = nhfr_side.count()
hfsrb_count = hfsrb_side.count()
unique_keys = id_map.select("facility_key").distinct().count()

print(f"ID map total rows: {id_map.count()}")
print(f"Unique facility_keys: {unique_keys}")
print(f"NHFR side rows: {nhfr_count}")
print(f"HFSRB side rows (excl duplicates): {hfsrb_count}")

In [0]:
# Build NHFR registry data: join id_map NHFR side to NHFR bronze on nhfr_id
nhfr_bronze = (
    spark.table(NHFR_BRONZE)
    .withColumn("_nhfr_id", F.regexp_extract(F.col("Health_Facility_Code"), r"^DOH(\d{15})$", 1).cast(LongType()))
)

nhfr_registry = (
    nhfr_side
    .join(nhfr_bronze, nhfr_side.nhfr_id == nhfr_bronze._nhfr_id, how="left")
    .select(
        F.col("facility_key"),
        F.col("Facility_Name").alias("nhfr_name"),
        F.col("Facility_Major_Type").alias("facility_major_type"),
        F.col("Health_Facility_Type").alias("health_facility_type"),
        F.col("Ownership_Major_Classification").alias("ownership_major_classification"),
        F.col("Ownership_Sub-Classification_for_Government_facilities").alias("ownership_sub_gov"),
        F.col("Ownership_Sub-Classification_for_private_facilities").alias("ownership_sub_priv"),
        F.col("Service_Capability").alias("nhfr_service_capability"),
        F.col("Bed_Capacity").alias("nhfr_bed_capacity_raw"),
        F.col("Licensing_Status").alias("nhfr_licensing_status"),
        F.col("Region_Name").alias("nhfr_region_name"),
        F.col("Region_PSGC").alias("nhfr_region_psgc"),
        F.col("Province_Name").alias("nhfr_province_name"),
        F.col("Province_PSGC").alias("nhfr_province_psgc"),
        F.col("City/Municipality_Name").alias("nhfr_municipality_name"),
        F.col("City/Municipality_PSGC").alias("nhfr_municipality_psgc"),
        F.col("Barangay_Name").alias("nhfr_barangay_name"),
        F.col("Barangay_PSGC").alias("nhfr_barangay_psgc"),
        F.col("_source_file").alias("nhfr_source_file"),
        F.col("_source_row_num").alias("nhfr_source_row_num"),
        F.col("_as_of_date").alias("nhfr_as_of_date"),
        F.col("_ingested_at").alias("nhfr_ingested_at"),
    )
)

# Verify all NHFR facility_keys matched to bronze
nhfr_matched = nhfr_registry.filter(F.col("nhfr_name").isNotNull()).count()
print(f"NHFR registry rows: {nhfr_registry.count()}")
print(f"NHFR matched to bronze: {nhfr_matched}")
assert nhfr_matched == nhfr_count, f"Expected {nhfr_count} NHFR rows to match bronze"
print("NHFR registry build complete")

In [0]:
# Build HFSRB registry data: join id_map HFSRB side to HFSRB bronze
# Join key: coalesce(license_no, source_file + "_" + source_row_num) = source_record_id
hfsrb_bronze = (
    spark.table(HFSRB_BRONZE)
    .withColumn(
        "_join_key",
        F.coalesce(
            F.col("license_no"),
            F.concat_ws("_", F.col("_source_file"), F.col("_source_row_num").cast("string"))
        )
    )
)

# For duplicate license_no: keep the row with latest validity_until per join_key
from pyspark.sql.window import Window

dup_window = Window.partitionBy("_join_key").orderBy(
    F.col("validity_until").desc_nulls_last(),
    F.col("_source_row_num").asc()
)

hfsrb_bronze_dedup = (
    hfsrb_bronze
    .withColumn("_dedup_rank", F.row_number().over(dup_window))
    .filter(F.col("_dedup_rank") == 1)
    .drop("_dedup_rank", "_nhfr_id")
)

hfsrb_registry = (
    hfsrb_side
    .join(hfsrb_bronze_dedup, hfsrb_side.hfsrb_source_record_id == hfsrb_bronze_dedup._join_key, how="left")
    .select(
        F.col("facility_key"),
        F.col("facility_name").alias("hfsrb_name"),
        F.col("license_no").alias("license_no"),
        F.col("ownership").alias("hfsrb_ownership"),
        F.col("ownership_class").alias("ownership_class"),
        F.col("hospital_class").alias("hospital_class"),
        F.col("service_capability").alias("hfsrb_service_capability"),
        F.col("abc").alias("hfsrb_abc_raw"),
        F.col("validity_from").alias("validity_from"),
        F.col("validity_until").alias("license_valid_until"),
        F.col("city_municipality").alias("hfsrb_municipality_name"),
        F.col("region").alias("hfsrb_region_name"),
        F.col("province").alias("hfsrb_province_name"),
        F.col("match_method"),
        F.col("match_status"),
        F.col("_source_file").alias("hfsrb_source_file"),
        F.col("_source_sheet").alias("hfsrb_source_sheet"),
        F.col("_source_row_num").alias("hfsrb_source_row_num"),
        F.col("_as_of_date").alias("hfsrb_as_of_date"),
        F.col("_ingested_at").alias("hfsrb_ingested_at"),
    )
)

hfsrb_matched_bronze = hfsrb_registry.filter(F.col("hfsrb_name").isNotNull()).count()
print(f"HFSRB registry rows: {hfsrb_registry.count()}")
print(f"HFSRB matched to bronze: {hfsrb_matched_bronze}")
print(f"HFSRB unmatched (name is null): {hfsrb_registry.filter(F.col('hfsrb_name').isNull()).count()}")

In [0]:
# Combine NHFR + HFSRB on facility_key (full outer join)
# One row per facility_key with both NHFR and HFSRB columns
registry = (
    nhfr_registry
    .join(hfsrb_registry, on="facility_key", how="full_outer")
)

total = registry.count()
in_both = registry.filter(F.col("nhfr_name").isNotNull() & F.col("hfsrb_name").isNotNull()).count()
nhfr_only = registry.filter(F.col("nhfr_name").isNotNull() & F.col("hfsrb_name").isNull()).count()
hfsrb_only = registry.filter(F.col("nhfr_name").isNull() & F.col("hfsrb_name").isNotNull()).count()
neither = registry.filter(F.col("nhfr_name").isNull() & F.col("hfsrb_name").isNull()).count()

print(f"Total registry rows: {total}")
print(f"  In both NHFR + HFSRB: {in_both}")
print(f"  NHFR only: {nhfr_only}")
print(f"  HFSRB only: {hfsrb_only}")
print(f"  Neither (unmatched): {neither}")
assert neither == 0, f"{neither} facility_keys have no source data"
print("Combine complete")

In [0]:
# Add derived columns: facility_name, service_capability, bed measures, presence/status, geography

# Parse bed capacity strings to integers (handle nulls, text, empty strings)
def safe_int(col):
    return F.when(col.isNotNull() & (F.trim(col) != ""), F.regexp_replace(F.trim(col), r"[^0-9]", "").cast("int")).otherwise(F.lit(None))

registry = (
    registry
    # Unified facility_name: prefer NHFR, fallback to HFSRB
    .withColumn("facility_name", F.coalesce(F.col("nhfr_name"), F.col("hfsrb_name")))
    .withColumn("facility_category", F.lit("hospital"))
    # Unified service_capability: prefer NHFR, fallback to HFSRB
    .withColumn("service_capability", F.coalesce(F.col("nhfr_service_capability"), F.col("hfsrb_service_capability")))
    # Bed measures (section 2.8)
    .withColumn("nhfr_reported_beds", safe_int(F.col("nhfr_bed_capacity_raw")))
    .withColumn("hfsrb_abc", safe_int(F.col("hfsrb_abc_raw")))
    .withColumn("authorized_beds", F.col("hfsrb_abc"))
    .withColumn("analysis_beds", F.coalesce(F.col("hfsrb_abc"), F.col("nhfr_reported_beds")))
    .withColumn(
        "analysis_beds_basis",
        F.when(F.col("hfsrb_abc").isNotNull(), F.lit("authorized"))
        .when(F.col("nhfr_reported_beds").isNotNull(), F.lit("nhfr_reported"))
        .otherwise(F.lit(None))
    )
    # Presence flags (section 2.7)
    .withColumn("is_currently_listed_nhfr", F.col("nhfr_name").isNotNull())
    .withColumn("is_currently_listed_hfsrb", F.col("hfsrb_name").isNotNull())
    .withColumn("in_nhfr", F.col("nhfr_name").isNotNull())
    .withColumn("in_hfsrb", F.col("hfsrb_name").isNotNull())
    .withColumn("facility_status", F.lit("listed"))
    # Geography: PSGC from NHFR (section 4.2 step 2)
    .withColumn("psgc_code", F.col("nhfr_municipality_psgc"))
    .withColumn(
        "geo_resolution_method",
        F.when(F.col("nhfr_municipality_psgc").isNotNull(), F.lit("source_code"))
        .otherwise(F.lit(None))
    )
    .withColumn("area_key", F.concat(F.lit("psgc:"), F.col("nhfr_municipality_psgc")))
    .withColumn(
        "municipality_name",
        F.coalesce(F.col("nhfr_municipality_name"), F.col("hfsrb_municipality_name"))
    )
    .withColumn("province_name", F.coalesce(F.col("nhfr_province_name"), F.col("hfsrb_province_name")))
    .withColumn("region_name", F.coalesce(F.col("nhfr_region_name"), F.col("hfsrb_region_name")))
    # Lineage
    .withColumn("_silver_build_id", F.lit(SILVER_BUILD_ID))
    .withColumn("_silver_loaded_at", F.current_timestamp())
    .withColumn("last_seen_at", F.current_timestamp())
)

# Summary
print("Derived columns added.")
print(f"  With analysis_beds: {registry.filter(F.col('analysis_beds').isNotNull()).count()}")
print(f"  With psgc_code: {registry.filter(F.col('psgc_code').isNotNull()).count()}")
print(f"  in_nhfr=True: {registry.filter(F.col('in_nhfr')).count()}")
print(f"  in_hfsrb=True: {registry.filter(F.col('in_hfsrb')).count()}")

In [0]:
# Select final columns and create target table
registry_final = registry.select(
    F.col("facility_key"),
    F.col("facility_name"),
    F.col("facility_category"),
    F.col("facility_major_type"),
    F.col("health_facility_type"),
    F.col("ownership_major_classification"),
    F.col("ownership_sub_gov"),
    F.col("ownership_sub_priv"),
    F.col("ownership_class"),
    F.col("service_capability"),
    F.col("hospital_class"),
    F.col("nhfr_licensing_status"),
    F.col("license_no"),
    F.col("license_valid_until"),
    F.col("validity_from"),
    # Bed measures
    F.col("nhfr_reported_beds"),
    F.col("hfsrb_abc"),
    F.col("authorized_beds"),
    F.col("analysis_beds"),
    F.col("analysis_beds_basis"),
    # Presence/status
    F.col("is_currently_listed_nhfr"),
    F.col("is_currently_listed_hfsrb"),
    F.col("in_nhfr"),
    F.col("in_hfsrb"),
    F.col("facility_status"),
    # Match info
    F.col("match_method"),
    # Geography
    F.col("area_key"),
    F.col("psgc_code"),
    F.col("municipality_name"),
    F.col("province_name"),
    F.col("region_name"),
    F.col("geo_resolution_method"),
    # Lineage - NHFR side
    F.col("nhfr_source_file"),
    F.col("nhfr_source_row_num"),
    F.col("nhfr_as_of_date"),
    F.col("nhfr_ingested_at"),
    # Lineage - HFSRB side
    F.col("hfsrb_source_file"),
    F.col("hfsrb_source_sheet"),
    F.col("hfsrb_source_row_num"),
    F.col("hfsrb_as_of_date"),
    F.col("hfsrb_ingested_at"),
    # Silver lineage
    F.col("_silver_build_id"),
    F.col("_silver_loaded_at"),
    F.col("last_seen_at"),
)

registry_final = registry_final.dropDuplicates(["facility_key"])

registry_final.createOrReplaceTempView("registry_candidate")

spark.sql("""
CREATE TABLE IF NOT EXISTS public_health.silver.facilities_registry_silver (
    facility_key                  STRING,
    facility_name                 STRING,
    facility_category             STRING,
    facility_major_type           STRING,
    health_facility_type          STRING,
    ownership_major_classification STRING,
    ownership_sub_gov             STRING,
    ownership_sub_priv            STRING,
    ownership_class               STRING,
    service_capability            STRING,
    hospital_class                STRING,
    nhfr_licensing_status         STRING,
    license_no                    STRING,
    license_valid_until           STRING,
    validity_from                 STRING,
    nhfr_reported_beds            INT,
    hfsrb_abc                     INT,
    authorized_beds              INT,
    analysis_beds                 INT,
    analysis_beds_basis           STRING,
    is_currently_listed_nhfr      BOOLEAN,
    is_currently_listed_hfsrb     BOOLEAN,
    in_nhfr                       BOOLEAN,
    in_hfsrb                      BOOLEAN,
    facility_status               STRING,
    match_method                  STRING,
    area_key                      STRING,
    psgc_code                     STRING,
    municipality_name             STRING,
    province_name                 STRING,
    region_name                   STRING,
    geo_resolution_method         STRING,
    nhfr_source_file              STRING,
    nhfr_source_row_num           INT,
    nhfr_as_of_date               DATE,
    nhfr_ingested_at              TIMESTAMP,
    hfsrb_source_file             STRING,
    hfsrb_source_sheet             STRING,
    hfsrb_source_row_num           INT,
    hfsrb_as_of_date               DATE,
    hfsrb_ingested_at              TIMESTAMP,
    _silver_build_id              STRING,
    _silver_loaded_at              TIMESTAMP,
    last_seen_at                  TIMESTAMP
)
USING DELTA
COMMENT 'Facility registry. Grain: 1 row per facility establishment. Key: facility_key.'
""")

print(f"Table ready: {TARGET_TABLE}")

In [0]:
# First build: CREATE OR REPLACE with deduplicated data (subsequent builds use MERGE for SCD-like updates)
spark.sql("""
CREATE OR REPLACE TABLE public_health.silver.facilities_registry_silver AS
SELECT * FROM registry_candidate
""")

print("Registry written (CREATE OR REPLACE)")

In [0]:
# Validation + summary
total = spark.table(TARGET_TABLE).count()
unique_keys = spark.table(TARGET_TABLE).select("facility_key").distinct().count()

print(f"Total registry rows: {total}")
print(f"Unique facility_keys: {unique_keys}")
if total != unique_keys:
    dup_count = total - unique_keys
    print(f"WARNING: {dup_count} duplicate facility_keys ({total} rows, {unique_keys} unique)")
    dup_keys = (
        spark.table(TARGET_TABLE)
        .groupBy("facility_key")
        .agg(F.count("*").alias("row_count"))
        .filter(F.col("row_count") > 1)
    )
    display(dup_keys.limit(20))
else:
    print("No duplicate facility_keys")

# Summary by source presence
print("\nSource presence:")
spark.sql("""
SELECT 
    in_nhfr, in_hfsrb, count(*) as row_count
FROM public_health.silver.facilities_registry_silver
GROUP BY in_nhfr, in_hfsrb
ORDER BY in_nhfr, in_hfsrb
""").show()

# Bed measures summary
print("Bed measures summary:")
spark.sql("""
SELECT
    analysis_beds_basis,
    count(*) as facilities,
    sum(analysis_beds) as total_beds,
    avg(analysis_beds) as avg_beds
FROM public_health.silver.facilities_registry_silver
WHERE analysis_beds IS NOT NULL
GROUP BY analysis_beds_basis
""").show()

# Geography resolution
print("Geography resolution:")
spark.sql("""
SELECT
    geo_resolution_method,
    count(*) as row_count
FROM public_health.silver.facilities_registry_silver
GROUP BY geo_resolution_method
""").show()

# Sample rows
print("Sample registry rows (matched facilities):")
spark.sql("""
SELECT facility_key, facility_name, ownership_major_classification, hospital_class,
       analysis_beds, analysis_beds_basis, psgc_code, match_method
FROM public_health.silver.facilities_registry_silver
WHERE in_nhfr AND in_hfsrb
LIMIT 10
""").show(truncate=False)

print("Sample HFSRB-only rows:")
spark.sql("""
SELECT facility_key, facility_name, ownership_class, hospital_class,
       hfsrb_abc, match_method
FROM public_health.silver.facilities_registry_silver
WHERE in_hfsrb AND NOT in_nhfr
LIMIT 10
""").show(truncate=False)

print("\nRegistry build complete.")